In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [78]:
from dotenv import load_dotenv
load_dotenv()

# let's import pretyprint
from pprint import pprint


True

In [48]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field
from langgraph.prebuilt import create_react_agent
from typing import Optional




In [14]:
# use langchain openai 
# Configure the client to use LM Studio's local server
client = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",  # LM Studio doesn't require a real API key,
    model="mistral-lmstudio",
    temperature=0,
)

In [106]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b

@tool
def write_email(content: str) -> str:
    """Writes an email to a friend."""
    return f"Hello, I wanted to share the following result with you: {content}. Best regards, Your friend."



In [107]:
response = client.invoke("First, add 100 and 42 and then multiply it by 5?")
print(response.content)

Sure! First, add 100 and 42:

\[ 100 + 42 = 142 \]

Next, multiply the result by 5:

\[ 142 \times 5 = 710 \]

So, the final answer is 710.


In [108]:
llm_with_tools = client.bind_tools([add_numbers, multiply_numbers], strict=True)

In [109]:
messages = [("user", "Multiply 50 with 2 and add 34 and 42")]
ai_msg = llm_with_tools.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '216675611', 'function': {'arguments': '{"a":50,"b":2}', 'name': 'multiply_numbers'}, 'type': 'function'}, {'id': '636721081', 'function': {'arguments': '{"a":34,"b":42}', 'name': 'add_numbers'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 257, 'total_tokens': 311, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen2.5-7b-instruct-1m', 'system_fingerprint': 'qwen2.5-7b-instruct-1m', 'id': 'chatcmpl-1ni0rmmlmblnrl83vbqhwe', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--c469c7ae-e5e5-4148-aee2-09a028d4bb77-0', tool_calls=[{'name': 'multiply_numbers', 'args': {'a': 50, 'b': 2}, 'id': '216675611', 'type': 'tool_call'}, {'name': 'add_numbers', 'args': {'a': 34, 'b': 42}, 'id': '636721081', 'type': 'tool_call'}], usage_metadata={'input_tokens': 257, 'output_tokens': 54, 'total_tokens': 311

In [135]:
class Output(BaseModel):
    '''Output from the llm'''
    result: list[str]


agent = create_react_agent(
    model=client,
    tools=[add_numbers, write_email],
    response_format=Output,  # Specify the output format
    debug=False,  # Enable debugging to see tool calls and responses

)


response = agent.invoke({"messages": [{"role": "user", 
                                       "content": "Add 100 and 42 and then write an email to XX@gmail.com with the result"}]})

In [136]:
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Add 100 and 42 and then write an email to XX@gmail.com with the result
================================== Ai Message ==================================
Tool Calls:
  add_numbers (894824052)
 Call ID: 894824052
  Args:
    a: 100
    b: 42
  write_email (677787292)
 Call ID: 677787292
  Args:
    content: The sum of 100 and 42 is 142.
================================= Tool Message =================================
Name: add_numbers

142
================================= Tool Message =================================
Name: write_email

Hello, I wanted to share the following result with you: The sum of 100 and 42 is 142.. Best regards, Your friend.
================================== Ai Message ==================================

The sum of 100 and 42 is 142. Here's an email that has been sent to XX@gmail.com:

Subject: Sum Result

Hello, I wanted to share the following result with you: The sum of 100 and 42 